In [16]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

deployment="gpt-3.5-turbo"

prompt1 = "advanced multi-agent course about azure in japanese"

messages = [{'role': 'user', 'content': prompt1}]

supported_locales = [
    "en-us",
    "zh-cn",
    "fr-ca",
    "ja-jp"
]

functions = [
   {
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            },
             "locale": {
                 "type":"string",
                 "description":f"A single, valid locale code from the supported list of locales {supported_locales}. The returned metadata will be in the requested locale if available. If this parameter isn't supplied, the en-us response will be returned."
             }
         },
         "required":[
            "role"
         ]
      }
   }
]

import requests

def search_courses(role, product, level, locale):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level,
        "locale": locale
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)

response = client.chat.completions.create(
    model=deployment,    
    messages = messages,
    functions = functions,
    function_call="auto"
)

print(response.choices[0])

Choice(finish_reason='function_call', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=FunctionCall(arguments='{"role":"developer","product":"Azure","level":"advanced","locale":"ja-jp"}', name='search_courses'), tool_calls=None))


In [17]:
# Check if the model wants to call a function
response_message = response.choices[0].message
if response_message.function_call.name:
    print("Recommended Function call:")
    print(response_message.function_call.name)
    print()

    # Call the function. 
    function_name = response_message.function_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(response_message.function_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the assistant response and function response to the messages
    messages.append( # adding assistant response to messages
        {
            "role": response_message.role,
            "function_call": {
                "name": function_name,
                "arguments": response_message.function_call.arguments,
            },
            "content": None
        }
    )
    messages.append( # adding function response to messages
        {
            "role": "function",
            "name": function_name,
            "content":function_response,
        }
    )

print("Messages in next request:")
print(messages)
print()

second_response = client.chat.completions.create(
    messages=messages,
    model=deployment,
    function_call="auto",
    functions=functions,
    temperature=0
)  # get a new response from GPT where it can see the function response

print(second_response.choices[0].message)

Recommended Function call:
search_courses

Output of function call:
[{'title': 'チーム内で知識を共有する', 'url': 'https://learn.microsoft.com/ja-jp/training/modules/share-knowledge-within-teams/?WT.mc_id=api_CatalogApi'}, {'title': '青緑色のデプロイと機能の切り替えを実装する', 'url': 'https://learn.microsoft.com/ja-jp/training/modules/implement-blue-green-deployment-feature-toggles/?WT.mc_id=api_CatalogApi'}, {'title': 'セキュリティの監視とガバナンス', 'url': 'https://learn.microsoft.com/ja-jp/training/modules/security-monitoring-and-governance/?WT.mc_id=api_CatalogApi'}, {'title': 'Azure Resource Manager テンプレートを使用して Azure リソースを作成する', 'url': 'https://learn.microsoft.com/ja-jp/training/modules/create-azure-resources-using-azure-resource-manager-templates/?WT.mc_id=api_CatalogApi'}, {'title': '開発者セルフサービスを実装する', 'url': 'https://learn.microsoft.com/ja-jp/training/modules/implement-developer-self-service/?WT.mc_id=api_CatalogApi'}]
<class 'str'>
Messages in next request:
[{'role': 'user', 'content': 'advanced multi-agent course about az